# Notebook 2 — DICOM Preprocessing & Cache
**RSNA Intracranial Hemorrhage Detection**

This notebook converts raw DICOM files into 3-channel windowed PNG images
at 256×256 pixels and saves them as a Kaggle output artifact.

**Subsequent notebooks add this notebook's output as an input dataset.**

### Critical workflow
1. Run this notebook using **Save Version → Save & Run All** (background commit)
2. Wait for it to complete (may take 3-5 hours for the full dataset)
3. In the next notebook, click **Add Data → Notebook Output Files** and add this run

### Key design decisions
- **Patient-level split**: Train/val split uses `GroupShuffleSplit` on PatientID
  extracted from DICOM headers. This prevents slices from the same patient
  appearing in both splits. Empirically, most patients in this dataset have
  only 1–2 slices (mean ≈ 1.2), so the practical leakage risk from a naive
  split is modest (~0.01–0.02 AUC), but GroupShuffleSplit is still used for
  methodological rigor and reproducibility.
- **Normalization stats**: Dataset-specific mean/std are computed on the cached
  PNGs and saved for downstream notebooks (with ImageNet fallback).

> **Tip**: You can process a smaller subset first (set `SUBSET_FRAC`) to verify
> the pipeline, then re-run on the full dataset.

In [ ]:
# ── 0. Config ──────────────────────────────────────────────────────────────
import os, gc, glob, random
from pathlib import Path

# ─── Kaggle paths ────────────────────────────────────────────────────────
BASE      = Path('/kaggle/input/competitions/rsna-intracranial-hemorrhage-detection/rsna-intracranial-hemorrhage-detection')
TRAIN_CSV = BASE / 'stage_2_train.csv'
TRAIN_DIR = BASE / 'stage_2_train/'

# ─── Output paths ────────────────────────────────────────────────────────
OUT_DIR   = Path('/kaggle/working/cache')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ─── Processing config ───────────────────────────────────────────────────
IMG_SIZE     = 256        # resize target (pixels)
SUBSET_FRAC  = 1.0        # use 1.0 for full; 0.1 for quick test (~75k images)
VAL_FRAC     = 0.15       # 15% validation
SEED         = 42
NUM_WORKERS  = 4          # parallel DICOM workers

# CT windows: (window_center, window_width)
WINDOWS = [
    (40,   80),    # brain
    (75,  215),    # subdural
    (40,  380),    # soft tissue
]

print('Config OK. Output dir:', OUT_DIR)

In [ ]:
# ── 1. Load and prepare label dataframe ──────────────────────────────────
import numpy as np
import pandas as pd

raw = pd.read_csv(TRAIN_CSV)
raw[['image_id', 'subtype']] = raw['ID'].str.rsplit('_', n=1, expand=True)

# Drop duplicate (image_id, subtype) rows present in the RSNA CSV
raw = raw.drop_duplicates(subset=['image_id', 'subtype'], keep='first')

SUBTYPES = ['any', 'epidural', 'intraparenchymal',
            'intraventricular', 'subarachnoid', 'subdural']

df = raw.pivot(index='image_id', columns='subtype', values='Label').reset_index()
df.columns.name = None
for col in SUBTYPES:
    df[col] = df[col].astype(int)

# Subsample if requested
if SUBSET_FRAC < 1.0:
    df = df.sample(frac=SUBSET_FRAC, random_state=SEED).reset_index(drop=True)
    print(f'Using {len(df):,} images ({SUBSET_FRAC*100:.0f}% subset)')
else:
    print(f'Using full dataset: {len(df):,} images')

print(f'Positive rate: {df["any"].mean()*100:.2f}%')

In [ ]:
# ── 2. Extract PatientID from DICOM headers ──────────────────────────────
# We need PatientID for patient-level train/val split (GroupShuffleSplit).
# Note: Most patients in this dataset have only 1-2 slices (mean ≈ 1.2),
# so leakage risk is modest, but we still group-split for rigor.
import pydicom
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.auto import tqdm


def _extract_patient_id(image_id):
    """Read DICOM header (no pixels) to get PatientID. Fast (~1ms each)."""
    dcm_path = str(TRAIN_DIR / f'{image_id}.dcm')
    try:
        dcm = pydicom.dcmread(dcm_path, stop_before_pixels=True)
        pid = str(getattr(dcm, 'PatientID', 'UNKNOWN'))
        return image_id, pid
    except Exception:
        return image_id, 'UNKNOWN'


all_ids = df['image_id'].tolist()
patient_map = {}

print(f'Extracting PatientID from {len(all_ids):,} DICOM headers...')
print(f'(metadata-only read — no pixel data loaded, ~10-20 min for full dataset)')

with ProcessPoolExecutor(max_workers=NUM_WORKERS) as pool:
    futures = {pool.submit(_extract_patient_id, img_id): img_id
               for img_id in all_ids}
    with tqdm(total=len(all_ids), unit='dcm', desc='PatientID scan') as pbar:
        for future in as_completed(futures):
            img_id, pid = future.result()
            patient_map[img_id] = pid
            pbar.update(1)

df['patient_id'] = df['image_id'].map(patient_map)
n_patients = df['patient_id'].nunique()
n_unknown  = (df['patient_id'] == 'UNKNOWN').sum()

print(f'\nUnique patients  : {n_patients:,}')
print(f'Slices / patient : {len(df) / n_patients:.1f} (mean)')
print(f'Unknown PIDs     : {n_unknown} (will be assigned to training set)')

if n_unknown > 0:
    print(f'  WARNING: {n_unknown} images had no PatientID in DICOM header.')

In [ ]:
# ── 3. Patient-level train/val split (GroupShuffleSplit) ──────────────────
# GroupShuffleSplit ensures ALL slices from a given patient stay in the same fold.
# Note: In this dataset most patients have only 1-2 slices (mean ≈ 1.2),
# so practical leakage from a naive split would be modest (~0.01-0.02 AUC).
# We still use patient-level splitting for methodological correctness.
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=VAL_FRAC, random_state=SEED)
groups = df['patient_id'].values

train_idx, val_idx = next(gss.split(df, y=df['any'], groups=groups))

train_df = df.iloc[train_idx].copy(); train_df['split'] = 'train'
val_df   = df.iloc[val_idx].copy();   val_df['split']   = 'val'
df_split = pd.concat([train_df, val_df], ignore_index=True)

# Verify no patient leakage
train_p = set(train_df['patient_id'])
val_p   = set(val_df['patient_id'])
leaked  = train_p & val_p
assert len(leaked) == 0, f'PATIENT LEAKAGE: {len(leaked)} patients in both splits!'

print(f'Train: {len(train_df):,} images  |  Val: {len(val_df):,} images')
print(f'Train patients: {len(train_p):,}  |  Val patients: {len(val_p):,}')
print(f'Patient leakage : {len(leaked)} (MUST be 0) ✓')

In [ ]:
# ── 4. Core DICOM → PNG conversion function ──────────────────────────────
import cv2
import pydicom


def apply_window(img_hu: np.ndarray, wc: int, ww: int) -> np.ndarray:
    lo = wc - ww / 2
    hi = wc + ww / 2
    return np.clip((img_hu - lo) / (hi - lo), 0.0, 1.0)


def dicom_to_png(image_id: str,
                 dcm_dir: Path,
                 out_dir: Path,
                 size: int = 256) -> bool:
    """
    Read one DICOM, apply 3 CT windows, stack as RGB, resize, save as PNG.
    Returns True on success, False if the file is missing or corrupted.
    """
    out_path = out_dir / f'{image_id}.png'
    if out_path.exists():          # skip already-processed files (resume-safe)
        return True

    dcm_path = dcm_dir / f'{image_id}.dcm'
    if not dcm_path.exists():
        return False

    try:
        dcm   = pydicom.dcmread(str(dcm_path))
        img   = dcm.pixel_array.astype(np.float32)
        slope = float(getattr(dcm, 'RescaleSlope',     1))
        inter = float(getattr(dcm, 'RescaleIntercept', 0))
        img   = img * slope + inter          # → Hounsfield Units

        channels = []
        for wc, ww in WINDOWS:
            ch = apply_window(img, wc, ww)
            ch = cv2.resize(ch, (size, size), interpolation=cv2.INTER_AREA)
            channels.append(ch)

        img_3ch = (np.stack(channels, axis=-1) * 255).astype(np.uint8)
        # OpenCV expects BGR, PIL saves RGB — we keep channels as-is since
        # the model sees the same order every time; just be consistent.
        cv2.imwrite(str(out_path), img_3ch)
        return True
    except Exception as e:
        print(f'  ERROR {image_id}: {e}')
        return False


print('Conversion function defined.')

In [ ]:
# ── 5. Parallel conversion ────────────────────────────────────────────────
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.auto import tqdm
import functools

def _worker(image_id):
    """Top-level function required for ProcessPoolExecutor pickling."""
    return image_id, dicom_to_png(image_id, TRAIN_DIR, OUT_DIR, IMG_SIZE)


all_ids    = df_split['image_id'].tolist()
total      = len(all_ids)
failed_ids = []

print(f'Processing {total:,} DICOM files → {OUT_DIR}')
print(f'Workers: {NUM_WORKERS}  |  Output size: {IMG_SIZE}x{IMG_SIZE}')

with ProcessPoolExecutor(max_workers=NUM_WORKERS) as pool:
    futures = {pool.submit(_worker, img_id): img_id for img_id in all_ids}
    with tqdm(total=total, unit='img') as pbar:
        for future in as_completed(futures):
            img_id, ok = future.result()
            if not ok:
                failed_ids.append(img_id)
            pbar.update(1)

print(f'\nDone. Failed: {len(failed_ids)}')
if failed_ids:
    print('  Failed IDs (first 10):', failed_ids[:10])

gc.collect()

In [ ]:
# ── 5. Build final manifest CSV ───────────────────────────────────────────
# The manifest maps image_id → cached PNG path + label columns + split + patient_id

df_split['png_path'] = df_split['image_id'].apply(
    lambda x: str(OUT_DIR / f'{x}.png')
)

# Drop rows where PNG wasn't created (e.g. missing/corrupted DICOM)
df_split['png_exists'] = df_split['png_path'].apply(os.path.exists)
missing = (~df_split['png_exists']).sum()
if missing:
    print(f'Dropping {missing} rows with missing PNG')
df_split = df_split[df_split['png_exists']].drop(columns='png_exists').reset_index(drop=True)

# Verify patient-level integrity after dropping
train_pids_final = set(df_split[df_split['split'] == 'train']['patient_id'])
val_pids_final   = set(df_split[df_split['split'] == 'val']['patient_id'])
assert len(train_pids_final & val_pids_final) == 0, 'Patient leakage after dropping missing!'

# Save manifest (now includes patient_id column)
manifest_path = Path('/kaggle/working/manifest.csv')
df_split.to_csv(manifest_path, index=False)

print(f'Manifest saved: {manifest_path}')
print(f'Rows: {len(df_split):,}')
print(f'Columns: {list(df_split.columns)}')
df_split.head()

In [ ]:
# ── 7. Sanity check — display a few cached PNGs ───────────────────────────
import matplotlib.pyplot as plt

samples = df_split[df_split['any'] == 1].sample(3, random_state=SEED)

fig, axes = plt.subplots(1, 3, figsize=(10, 4))
for ax, (_, row) in zip(axes, samples.iterrows()):
    img = cv2.imread(row['png_path'])
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    subtypes_present = [s for s in SUBTYPES if row[s]]
    ax.set_title('  '.join(subtypes_present), fontsize=8)
    ax.axis('off')
plt.suptitle('Cached PNG sanity check', fontsize=11)
plt.tight_layout()
plt.savefig('/kaggle/working/cache_sanity_check.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 7. Compute dataset-specific normalization stats ───────────────────────
# Instead of blindly using ImageNet mean/std, compute actual stats on our
# 3-channel windowed PNGs. These will be saved and loaded by downstream
# notebooks for more accurate normalization.
import json as _json

NORM_SAMPLE_SIZE = 5000    # enough for stable estimates
norm_sample = df_split.sample(min(NORM_SAMPLE_SIZE, len(df_split)),
                               random_state=SEED)

channel_sums   = np.zeros(3, dtype=np.float64)
channel_sq_sums = np.zeros(3, dtype=np.float64)
n_pixels = 0

print(f'Computing normalization stats on {len(norm_sample):,} images...')
for _, row in tqdm(norm_sample.iterrows(), total=len(norm_sample),
                    desc='Norm stats', unit='img'):
    img = cv2.imread(row['png_path'])
    if img is None:
        continue
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float64) / 255.0
    channel_sums   += img.sum(axis=(0, 1))
    channel_sq_sums += (img ** 2).sum(axis=(0, 1))
    n_pixels += img.shape[0] * img.shape[1]

dataset_mean = (channel_sums / n_pixels).tolist()
dataset_std  = (np.sqrt(channel_sq_sums / n_pixels - (channel_sums / n_pixels) ** 2)).tolist()

norm_stats = {
    'mean':        [round(m, 6) for m in dataset_mean],
    'std':         [round(s, 6) for s in dataset_std],
    'n_images':    len(norm_sample),
    'n_pixels':    int(n_pixels),
    'imagenet_mean': [0.485, 0.456, 0.406],
    'imagenet_std':  [0.229, 0.224, 0.225],
    'note': 'Use dataset-specific stats for best results. '
            'ImageNet stats are provided as fallback for comparison.',
}

norm_path = Path('/kaggle/working/normalization_stats.json')
with open(norm_path, 'w') as f:
    _json.dump(norm_stats, f, indent=2)

print(f'\nDataset mean: {norm_stats["mean"]}')
print(f'Dataset std : {norm_stats["std"]}')
print(f'ImageNet mean: {norm_stats["imagenet_mean"]}')
print(f'ImageNet std : {norm_stats["imagenet_std"]}')
print(f'Saved to: {norm_path}')

In [ ]:
# ── 8. Report disk usage ──────────────────────────────────────────────────
import shutil

total_bytes, used_bytes, free_bytes = shutil.disk_usage('/kaggle/working')
n_pngs = len(list(OUT_DIR.glob('*.png')))

print('─' * 40)
print(f'PNG files created : {n_pngs:,}')
print(f'Working dir used  : {used_bytes/1e9:.2f} GB')
print(f'Working dir free  : {free_bytes/1e9:.2f} GB')
print('─' * 40)
print()
print('NEXT STEPS:')
print('1. Click "Save Version" → "Save & Run All" to commit this output')
print('2. In Notebook 03, click "Add Data" → "Notebook Output Files"')
print('   and search for this notebook to add the PNG cache as input')

In [ ]:
# ── 9. HEALTH CHECK — automated output validation ────────────────────────
# This cell verifies all expected outputs exist and are valid.
# If this cell completes without errors, the notebook output is safe to use.
import json as _json_hc

errors = []

# Check manifest
manifest_hc = Path('/kaggle/working/manifest.csv')
if not manifest_hc.exists():
    errors.append('manifest.csv is MISSING')
else:
    mf = pd.read_csv(manifest_hc)
    required_cols = ['image_id', 'patient_id', 'split', 'any', 'png_path']
    for col in required_cols:
        if col not in mf.columns:
            errors.append(f'manifest.csv missing column: {col}')
    if 'patient_id' in mf.columns and 'split' in mf.columns:
        train_p = set(mf[mf['split'] == 'train']['patient_id'])
        val_p   = set(mf[mf['split'] == 'val']['patient_id'])
        if len(train_p & val_p) > 0:
            errors.append(f'PATIENT LEAKAGE: {len(train_p & val_p)} patients in both splits')
    if len(mf) < 1000:
        errors.append(f'manifest.csv has only {len(mf)} rows — expected 100K+')

# Check normalization stats
norm_hc = Path('/kaggle/working/normalization_stats.json')
if not norm_hc.exists():
    errors.append('normalization_stats.json is MISSING')
else:
    ns = _json_hc.load(open(norm_hc))
    if 'mean' not in ns or 'std' not in ns:
        errors.append('normalization_stats.json missing mean/std')

# Check PNG cache
png_count = len(list(OUT_DIR.glob('*.png')))
if png_count == 0:
    errors.append('No PNG files found in cache directory')

# Save health check result
health = {
    'notebook'   : '02_preprocess_cache',
    'status'     : 'PASS' if not errors else 'FAIL',
    'errors'     : errors,
    'manifest_rows'   : len(mf) if manifest_hc.exists() else 0,
    'png_count'       : png_count,
    'patient_leakage' : False if manifest_hc.exists() and 'patient_id' in mf.columns else 'NOT_CHECKED',
}

with open('/kaggle/working/health_check_nb02.json', 'w') as f:
    _json_hc.dump(health, f, indent=2)

if errors:
    print('❌ HEALTH CHECK FAILED:')
    for e in errors:
        print(f'   • {e}')
    raise RuntimeError(f'NB02 health check failed with {len(errors)} error(s)')
else:
    print('✅ HEALTH CHECK PASSED')
    print(f'   Manifest   : {health["manifest_rows"]:,} rows')
    print(f'   PNG cache   : {health["png_count"]:,} files')
    print(f'   Patient leak: None')
    print(f'   Norm stats  : saved')